# Avito Services — Candidate Generation

Цель задачи — для каждого поискового запроса выбрать до 50 объявлений
из общего корпуса с максимальным Recall@50.

Итоговое решение состоит из двух этапов:

1. Candidate retrieval:
   - fine-tuned multilingual E5;
   - word-level TF-IDF;
   - character-level TF-IDF;
   - TF-IDF с использованием описания объявления;
   - location prior, рассчитанный по train.

2. Pre-ranking:
   - объединение кандидатов из retrieval-каналов;
   - предварительный отбор top-300;
   - CatBoostRanker по признакам retrieval-каналов, semantic similarity,
     location и признакам объявления;
   - выбор итоговых top-50 кандидатов.

Финальный результат на скрытом benchmark:
**Recall@50 = 0.818530**.

In [14]:
from pathlib import Path
import random
import time

import numpy as np
import pandas as pd

import torch

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold

from sentence_transformers import SentenceTransformer
from catboost import CatBoostRanker, Pool

In [15]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path("dataset")

TRAIN_PATH = DATA_DIR / "train.parquet"
QUERIES_PATH = DATA_DIR / "benchmark_queries.parquet"
ITEMS_PATH = DATA_DIR / "benchmark_items.parquet"

FINAL_MODEL_PATH = Path("e5_avito_pilot2_final")
ANSWER_PATH = Path("answer.csv")

DEVICE = (
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", DEVICE)

Device: mps


In [16]:
train = pd.read_parquet(TRAIN_PATH)
benchmark_queries = pd.read_parquet(QUERIES_PATH)
items = pd.read_parquet(ITEMS_PATH)

print("train:", train.shape)
print("benchmark_queries:", benchmark_queries.shape)
print("items:", items.shape)

train: (497673, 19)
benchmark_queries: (2452, 6)
items: (189212, 14)


## Используемые данные

### Query-признаки

- `search_query` — текст поискового запроса;
- `search_infm_params_text` — структурированные параметры поиска;
- `search_location_id` — географический идентификатор поиска.

### Item-признаки

Для retrieval используются:

- `item_title_raw`;
- `item_infm_params_text`;
- `item_description_raw`.

Для pre-ranking дополнительно используются:

- `item_location_id`;
- `item_rating`;
- `item_rating_reviews_count`;
- `item_price`;
- `item_is_phone_hidden`;
- `item_is_message_forbidden`.

Категория не использовалась как жёсткий фильтр: почти весь benchmark-корпус
принадлежит одной основной категории, поэтому такой фильтр практически не
сокращает множество кандидатов.

Location также не используется как жёсткий фильтр, поскольку часть релевантных
объявлений находится в соседних или дочерних location. Вместо этого используется
вероятностный location prior.

## Локальная валидация

Для оценки качества до отправки сформирован фиксированный holdout из 2452 запросов —
того же размера, что и benchmark.

В validation попадают только пары `(query, item)`, для которых `item_id`
присутствует в `benchmark_items`, поскольку retrieval выполняется именно по этому корпусу.

Validation состоит из двух частей:

- **seen (907 запросов)** — запрос остаётся представлен в обучающей выборке другими
  положительными item; для этого выбираются только запросы, имеющие не менее двух
  различных положительных `item_id`;
- **unseen (1545 запросов)** — все строки соответствующего `search_query`
  полностью удаляются из обучающей части.

Для каждого validation-запроса скрывается одна положительная пара.
Фиксированный `random_state=25` делает split воспроизводимым.

In [17]:
SEED = 42  # обучение / остальные эксперименты
SPLIT_RANDOM_STATE = 25  # фиксированный validation split

N_SEEN = 907
N_UNSEEN = 1545

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [18]:
rng = np.random.default_rng(SPLIT_RANDOM_STATE)

# В retrieval можно найти только item из benchmark-корпуса.
benchmark_item_ids = set(items["item_id"])

available = train[
    train["item_id"].isin(benchmark_item_ids)
].copy()

print("Доступных positive-строк:", len(available))
print("Уникальных search_query:", available["search_query"].nunique())

# Сколько различных positive item есть у каждого текста запроса.
query_unique_items = (
    train
    .groupby("search_query")["item_id"]
    .nunique()
)

# Для seen нужен минимум второй positive:
# один item скрываем для validation,
# другой остаётся в обучающей части.
seen_candidates = (
    available["search_query"]
    .drop_duplicates()
)

seen_candidates = seen_candidates[
    seen_candidates.map(query_unique_items) >= 2
    ].to_numpy()

print("Кандидатов для seen:", len(seen_candidates))

Доступных positive-строк: 33010
Уникальных search_query: 12208
Кандидатов для seen: 8893


In [19]:
seen_queries = rng.choice(
    seen_candidates,
    size=N_SEEN,
    replace=False,
)

seen_query_set = set(seen_queries)

unseen_candidates = (
    available.loc[
        ~available["search_query"].isin(seen_query_set),
        "search_query",
    ]
    .drop_duplicates()
    .to_numpy()
)

assert len(unseen_candidates) >= N_UNSEEN

unseen_queries = rng.choice(
    unseen_candidates,
    size=N_UNSEEN,
    replace=False,
)

unseen_query_set = set(unseen_queries)

assert len(seen_query_set & unseen_query_set) == 0

print("seen:", len(seen_query_set))
print("unseen:", len(unseen_query_set))

seen: 907
unseen: 1545


In [20]:
selected_queries = seen_query_set | unseen_query_set

val_pool = available[
    available["search_query"].isin(selected_queries)
].copy()

# Берём ровно одну positive-строку на search_query.
validation = (
    val_pool
    .sample(frac=1, random_state=SPLIT_RANDOM_STATE)
    .drop_duplicates("search_query")
    .copy()
)

validation["split"] = np.where(
    validation["search_query"].isin(seen_query_set),
    "seen",
    "unseen",
)

validation = validation.rename(
    columns={"item_id": "relevant_item_id"}
)

validation = validation[
    [
        "search_query",
        "search_location_id",
        "search_is_delivery_search",
        "search_infm_params_text",
        "search_category",
        "relevant_item_id",
        "split",
    ]
].reset_index(drop=True)

assert validation.shape[0] == 2452

print(validation.shape)
print()
print(validation["split"].value_counts())

validation.head()

(2452, 7)

split
unseen    1545
seen       907
Name: count, dtype: int64


,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,relevant_item_id,split
0,замена аккумулятора iphone 12,637640,0,Тип услуги Мобильные устройства Вид услуги Рем...,114,2ba31249ca4ea30e,seen
1,кадастровый инженер,662810,0,Вид услуги Деловые услуги,114,18c0b8a44bbf61e2,unseen
2,увеличение губ,653240,0,"Тип услуги Косметология Вид услуги Красота, зд...",114,391037f10a20ee24,unseen
3,перетяжка мебели,658150,0,,114,57880b4e0ad0421c,seen
4,грузоперевозки зил,621800,0,,114,871d9d8b4b573adc,unseen


## Формирование обучающей части

Чтобы локальная оценка не содержала утечки:

- все `unseen` query полностью удаляются из обучающей выборки;
- для `seen` query удаляется только скрытая положительная пара
  `(search_query, relevant_item_id)`;
- остальные положительные примеры `seen`-запроса остаются в train.

Таким образом, модель не видит validation-positive при обучении,
но для `seen`-части сохраняется реалистичный сценарий повторяющихся запросов.

In [21]:
unseen_queries = set(
    validation.loc[
        validation["split"] == "unseen",
        "search_query",
    ]
)

seen_hidden_pairs = (
    validation.loc[
        validation["split"] == "seen",
        ["search_query", "relevant_item_id"],
    ]
    .rename(columns={"relevant_item_id": "item_id"})
    .drop_duplicates()
)

print("Unseen query texts:", len(unseen_queries))
print("Seen hidden pairs:", len(seen_hidden_pairs))

Unseen query texts: 1545
Seen hidden pairs: 907


## Dense retrieval: fine-tuned multilingual E5

В качестве основного семантического retriever используется
`intfloat/multilingual-e5-small`, дополнительно обученный на положительных
query-item парах из train.

Для E5 используются стандартные префиксы:

- запрос: `query: <search_query>. <search_infm_params_text>`
- объявление: `passage: <item_title_raw>. <item_infm_params_text>`

Описание объявления (`item_description_raw`) в dense-модель не передаётся:
оно используется отдельно в lexical retrieval.

Fine-tuning модели вынесен в отдельный notebook. Для быстрого воспроизведения
итогового решения используется сохранённый checkpoint `e5_avito_pilot2_final`.

Embeddings L2-нормализованы, поэтому их скалярное произведение соответствует
cosine similarity.

In [22]:
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def build_query(row):
    query = clean_text(row["search_query"])
    params = clean_text(row["search_infm_params_text"])

    if params:
        return f"query: {query}. {params}"

    return f"query: {query}"


def build_item(row):
    title = clean_text(row["item_title_raw"])
    params = clean_text(row["item_infm_params_text"])

    if params:
        return f"passage: {title}. {params}"

    return f"passage: {title}"

In [23]:
from sentence_transformers import SentenceTransformer

MODEL_PATH = Path("e5_avito_pilot2_final")

assert MODEL_PATH.exists(), (
    "Не найден checkpoint e5_avito_pilot2_final"
)

model_final = SentenceTransformer(
    str(MODEL_PATH),
    device=DEVICE,
)

model_final.max_seq_length = 128

print("Dense model loaded.")
print("Device:", DEVICE)

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 8736.49it/s]


Dense model loaded.
Device: mps


In [24]:
item_texts = (
    items
    .apply(build_item, axis=1)
    .tolist()
)

print("Items:", len(item_texts))
print(item_texts[:2])

Items: 189212
['passage: Ремонт/выкуп компьют. и ноутбуков с выездом на дом. Вид услуги Компьютерная помощь Место оказания услуг пр-т Ленина Тип стоимости за услугу Начальная цена График работы от 34200 График работы до 82800 Время работы, с 09:30 Время работы, до 23:00 Начальная цена Тип стоимости за услугу Стоимость 500 Услуга Диагностика компьютера Начальная цена Тип стоимости за услугу Стоимость 500 Услуга Начальная цена Тип стоимости за услугу Стоимость 500 Услуга Начальная цена Тип стоимости за услугу Стоимость 1000 Услуга Замена термопасты Начальная цена Тип стоимости за услугу Стоимость 1000 Услуга Ремонт после залития Начальная цена Тип стоимости за услугу Стоимость 500 Услуга Чистка системы охлаждения Начальная цена Тип стоимости за услугу Стоимость 500 Услуга Установка базовых драйверов Начальная цена Тип стоимости за услугу Стоимость 1000 Услуга Восстановление удалённых файлов Начальная цена Тип стоимости за услугу Стоимость 500 Услуга Установка антивируса Начальная цена Ти

In [25]:
ITEM_EMBEDDINGS_PATH = Path(
    "artifacts/item_embeddings_e5_avito_pilot2.npy"
)

if ITEM_EMBEDDINGS_PATH.exists():

    item_embeddings_ft2 = np.load(
        ITEM_EMBEDDINGS_PATH
    )

    print("Loaded cached item embeddings.")

else:

    start = time.time()

    item_embeddings_ft2 = model_final.encode(
        item_texts,
        batch_size=256,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    np.save(
        ITEM_EMBEDDINGS_PATH,
        item_embeddings_ft2,
    )

    print(
        f"Computed in "
        f"{(time.time() - start) / 60:.1f} min"
    )

print(
    item_embeddings_ft2.shape,
    item_embeddings_ft2.dtype,
)

Loaded cached item embeddings.
(189212, 384) float32


In [26]:
val_query_texts = (
    validation
    .apply(build_query, axis=1)
    .tolist()
)

VAL_EMBEDDINGS_PATH = Path(
    "artifacts/val_embeddings_e5_avito_pilot2.npy"
)

if VAL_EMBEDDINGS_PATH.exists():

    val_embeddings_ft2 = np.load(
        VAL_EMBEDDINGS_PATH
    )

    print("Loaded cached validation embeddings.")

else:

    val_embeddings_ft2 = model_final.encode(
        val_query_texts,
        batch_size=256,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    np.save(
        VAL_EMBEDDINGS_PATH,
        val_embeddings_ft2,
    )

print(
    val_embeddings_ft2.shape,
    val_embeddings_ft2.dtype,
)

Loaded cached validation embeddings.
(2452, 384) float32


In [27]:
benchmark_query_texts = (
    benchmark_queries
    .apply(build_query, axis=1)
    .tolist()
)

BENCHMARK_EMBEDDINGS_PATH = Path(
    "artifacts/benchmark_query_embeddings_ft2.npy"
)

if BENCHMARK_EMBEDDINGS_PATH.exists():

    benchmark_embeddings_ft2 = np.load(
        BENCHMARK_EMBEDDINGS_PATH
    )

    print("Loaded cached benchmark embeddings.")

else:

    benchmark_embeddings_ft2 = model_final.encode(
        benchmark_query_texts,
        batch_size=256,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    np.save(
        BENCHMARK_EMBEDDINGS_PATH,
        benchmark_embeddings_ft2,
    )

print(
    benchmark_embeddings_ft2.shape,
    benchmark_embeddings_ft2.dtype,
)

Loaded cached benchmark embeddings.
(2452, 384) float32


## Dense candidate retrieval

Для каждого запроса выбираются top-1000 объявлений по cosine similarity
между E5 embeddings.

Так как embeddings предварительно L2-нормализованы, cosine similarity
вычисляется обычным скалярным произведением.

На текущем размере корпуса (~189 тыс. объявлений) точный поиск через PyTorch
достаточно быстрый и позволяет избежать дополнительной зависимости от ANN/FAISS.

In [28]:
def dense_topk(
        query_embeddings,
        item_embeddings,
        k=1000,
        batch_size=20,
        device=DEVICE,
):
    item_tensor = torch.from_numpy(
        item_embeddings
    ).to(device)

    all_scores = []
    all_indices = []

    start_total = time.time()

    with torch.no_grad():
        for start in range(
                0,
                len(query_embeddings),
                batch_size,
        ):
            end = min(
                start + batch_size,
                len(query_embeddings),
            )

            q = torch.from_numpy(
                query_embeddings[start:end]
            ).to(device)

            similarities = q @ item_tensor.T

            top_scores, top_indices = torch.topk(
                similarities,
                k=k,
                dim=1,
                largest=True,
                sorted=True,
            )

            all_scores.append(
                top_scores.cpu().numpy()
            )

            all_indices.append(
                top_indices.cpu().numpy()
            )

            del q, similarities, top_scores, top_indices

            if end % 200 == 0 or end == len(query_embeddings):
                print(
                    f"{end}/{len(query_embeddings)} "
                    f"— {time.time() - start_total:.1f}s"
                )

    scores = np.vstack(all_scores).astype(np.float32)
    indices = np.vstack(all_indices).astype(np.int32)

    return scores, indices

In [29]:
VAL_DENSE_SCORES_PATH = Path(
    "artifacts/dense_e5_avito_pilot2_scores_top1000.npy"
)

VAL_DENSE_INDICES_PATH = Path(
    "artifacts/dense_e5_avito_pilot2_indices_top1000.npy"
)

if (
        VAL_DENSE_SCORES_PATH.exists()
        and VAL_DENSE_INDICES_PATH.exists()
):
    scores_ft2_1000 = np.load(
        VAL_DENSE_SCORES_PATH
    )

    indices_ft2_1000 = np.load(
        VAL_DENSE_INDICES_PATH
    )

    print("Loaded cached validation dense top-1000.")

else:
    scores_ft2_1000, indices_ft2_1000 = dense_topk(
        query_embeddings=val_embeddings_ft2,
        item_embeddings=item_embeddings_ft2,
        k=1000,
    )

    np.save(
        VAL_DENSE_SCORES_PATH,
        scores_ft2_1000,
    )

    np.save(
        VAL_DENSE_INDICES_PATH,
        indices_ft2_1000,
    )

print(scores_ft2_1000.shape)
print(indices_ft2_1000.shape)

Loaded cached validation dense top-1000.
(2452, 1000)
(2452, 1000)


In [30]:
VAL_DENSE_SCORES_PATH = Path(
    "artifacts/dense_e5_avito_pilot2_scores_top1000.npy"
)

VAL_DENSE_INDICES_PATH = Path(
    "artifacts/dense_e5_avito_pilot2_indices_top1000.npy"
)

if (
        VAL_DENSE_SCORES_PATH.exists()
        and VAL_DENSE_INDICES_PATH.exists()
):
    scores_ft2_1000 = np.load(
        VAL_DENSE_SCORES_PATH
    )

    indices_ft2_1000 = np.load(
        VAL_DENSE_INDICES_PATH
    )

    print("Loaded cached validation dense top-1000.")

else:
    scores_ft2_1000, indices_ft2_1000 = dense_topk(
        query_embeddings=val_embeddings_ft2,
        item_embeddings=item_embeddings_ft2,
        k=1000,
    )

    np.save(
        VAL_DENSE_SCORES_PATH,
        scores_ft2_1000,
    )

    np.save(
        VAL_DENSE_INDICES_PATH,
        indices_ft2_1000,
    )

print(scores_ft2_1000.shape)
print(indices_ft2_1000.shape)

Loaded cached validation dense top-1000.
(2452, 1000)
(2452, 1000)


In [31]:
item_ids = (
    items["item_id"]
    .astype(str)
    .to_numpy()
)

relevant = (
    validation["relevant_item_id"]
    .astype(str)
    .to_numpy()
)

dense_top1000_ids = item_ids[
    indices_ft2_1000
]

dense_hits_1000 = np.array([
    relevant[i] in dense_top1000_ids[i]
    for i in range(len(validation))
])

print(
    "Dense Recall@1000:",
    f"{dense_hits_1000.mean():.4f}",
    f"({dense_hits_1000.sum()}/{len(dense_hits_1000)})",
)

Dense Recall@1000: 0.8613 (2112/2452)


In [32]:
BENCH_DENSE_SCORES_PATH = Path(
    "artifacts/benchmark_ft2_dense_scores_top1000.npy"
)

BENCH_DENSE_INDICES_PATH = Path(
    "artifacts/benchmark_ft2_dense_indices_top1000.npy"
)

if (
        BENCH_DENSE_SCORES_PATH.exists()
        and BENCH_DENSE_INDICES_PATH.exists()
):
    benchmark_dense_scores = np.load(
        BENCH_DENSE_SCORES_PATH
    )

    benchmark_dense_indices = np.load(
        BENCH_DENSE_INDICES_PATH
    )

    print("Loaded cached benchmark dense top-1000.")

else:
    benchmark_dense_scores, benchmark_dense_indices = dense_topk(
        query_embeddings=benchmark_embeddings_ft2,
        item_embeddings=item_embeddings_ft2,
        k=1000,
    )

    np.save(
        BENCH_DENSE_SCORES_PATH,
        benchmark_dense_scores,
    )

    np.save(
        BENCH_DENSE_INDICES_PATH,
        benchmark_dense_indices,
    )

print(benchmark_dense_scores.shape)
print(benchmark_dense_indices.shape)

Loaded cached benchmark dense top-1000.
(2452, 1000)
(2452, 1000)


In [33]:
def sparse_topk(
        X_queries,
        X_items,
        k=1000,
        batch_size=10,
):
    all_indices = []
    all_scores = []

    start_total = time.time()

    for start in range(0, X_queries.shape[0], batch_size):
        end = min(
            start + batch_size,
            X_queries.shape[0],
        )

        similarities = (
                X_queries[start:end] @ X_items.T
        ).toarray()

        top_idx = np.argpartition(
            -similarities,
            kth=k - 1,
            axis=1,
        )[:, :k]

        top_scores = np.take_along_axis(
            similarities,
            top_idx,
            axis=1,
        )

        order = np.argsort(
            -top_scores,
            axis=1,
        )

        top_idx = np.take_along_axis(
            top_idx,
            order,
            axis=1,
        )

        top_scores = np.take_along_axis(
            top_scores,
            order,
            axis=1,
        )

        all_indices.append(top_idx)
        all_scores.append(top_scores)

        if end % 200 == 0 or end == X_queries.shape[0]:
            print(
                f"{end}/{X_queries.shape[0]} "
                f"— {time.time() - start_total:.1f}s"
            )

    return (
        np.vstack(all_indices).astype(np.int32),
        np.vstack(all_scores).astype(np.float32),
    )

## Lexical retrieval

Dense retrieval дополняется тремя независимыми lexical-каналами:

1. word-level TF-IDF по `title + params`;
2. character-level TF-IDF по тому же тексту;
3. word-level TF-IDF с добавлением `description`.

Эти каналы полезны для точных совпадений терминов, опечаток, моделей,
названий услуг и редких слов, которые dense retriever может ранжировать хуже.

In [34]:
item_texts_lex = (
        items["item_title_raw"].fillna("").astype(str)
        + " "
        + items["item_infm_params_text"].fillna("").astype(str)
)

val_texts_lex = (
        validation["search_query"].fillna("").astype(str)
        + " "
        + validation["search_infm_params_text"].fillna("").astype(str)
)

benchmark_texts_lex = (
        benchmark_queries["search_query"].fillna("").astype(str)
        + " "
        + benchmark_queries["search_infm_params_text"].fillna("").astype(str)
)

In [35]:
word_vectorizer = TfidfVectorizer(
    lowercase=True,
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_features=200_000,
    sublinear_tf=True,
    norm="l2",
    dtype=np.float32,
    token_pattern=r"(?u)\b\w+\b",
)

In [36]:
VAL_WORD_PATH = Path(
    "artifacts/lexical_tfidf_indices_top1000.npy"
)

BENCH_WORD_PATH = Path(
    "artifacts/benchmark_lexical_indices_top1000.npy"
)

if VAL_WORD_PATH.exists() and BENCH_WORD_PATH.exists():

    lex_indices_1000 = np.load(
        VAL_WORD_PATH
    )

    benchmark_lex_indices = np.load(
        BENCH_WORD_PATH
    )

    print("Loaded cached word TF-IDF retrieval.")

else:

    X_items_word = word_vectorizer.fit_transform(
        item_texts_lex
    )

    X_val_word = word_vectorizer.transform(
        val_texts_lex
    )

    X_bench_word = word_vectorizer.transform(
        benchmark_texts_lex
    )

    lex_indices_1000, _ = sparse_topk(
        X_val_word,
        X_items_word,
    )

    benchmark_lex_indices, _ = sparse_topk(
        X_bench_word,
        X_items_word,
    )

    np.save(
        VAL_WORD_PATH,
        lex_indices_1000,
    )

    np.save(
        BENCH_WORD_PATH,
        benchmark_lex_indices,
    )

print(lex_indices_1000.shape)
print(benchmark_lex_indices.shape)

Loaded cached word TF-IDF retrieval.
(2452, 1000)
(2452, 1000)


In [37]:
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_features=150_000,
    sublinear_tf=True,
    norm="l2",
    dtype=np.float32,
)

In [38]:
VAL_CHAR_PATH = Path(
    "artifacts/char_tfidf_indices_top1000.npy"
)

BENCH_CHAR_PATH = Path(
    "artifacts/benchmark_char_tfidf_indices_top1000.npy"
)

if VAL_CHAR_PATH.exists() and BENCH_CHAR_PATH.exists():

    char_indices_1000 = np.load(
        VAL_CHAR_PATH
    )

    benchmark_char_indices = np.load(
        BENCH_CHAR_PATH
    )

    print("Loaded cached char TF-IDF retrieval.")

else:

    X_items_char = char_vectorizer.fit_transform(
        item_texts_lex
    )

    X_val_char = char_vectorizer.transform(
        val_texts_lex
    )

    X_bench_char = char_vectorizer.transform(
        benchmark_texts_lex
    )

    char_indices_1000, _ = sparse_topk(
        X_val_char,
        X_items_char,
    )

    benchmark_char_indices, _ = sparse_topk(
        X_bench_char,
        X_items_char,
    )

    np.save(
        VAL_CHAR_PATH,
        char_indices_1000,
    )

    np.save(
        BENCH_CHAR_PATH,
        benchmark_char_indices,
    )

print(char_indices_1000.shape)
print(benchmark_char_indices.shape)

Loaded cached char TF-IDF retrieval.
(2452, 1000)
(2452, 1000)


In [39]:
item_texts_desc = (
        items["item_title_raw"].fillna("").astype(str)
        + " "
        + items["item_infm_params_text"].fillna("").astype(str)
        + " "
        + items["item_description_raw"]
        .fillna("")
        .astype(str)
        .str.slice(0, 1500)
)

In [40]:
val_texts_desc = val_texts_lex
benchmark_texts_desc = benchmark_texts_lex

In [41]:
desc_vectorizer = TfidfVectorizer(
    lowercase=True,
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_features=200_000,
    sublinear_tf=True,
    norm="l2",
    dtype=np.float32,
    token_pattern=r"(?u)\b\w+\b",
)

In [42]:
VAL_DESC_PATH = Path(
    "artifacts/description_tfidf_indices_top1000.npy"
)

BENCH_DESC_PATH = Path(
    "artifacts/benchmark_description_tfidf_indices_top1000.npy"
)

if VAL_DESC_PATH.exists() and BENCH_DESC_PATH.exists():

    desc_indices_1000 = np.load(
        VAL_DESC_PATH
    )

    benchmark_desc_indices = np.load(
        BENCH_DESC_PATH
    )

    print("Loaded cached description TF-IDF retrieval.")

else:

    X_items_desc = desc_vectorizer.fit_transform(
        item_texts_desc
    )

    X_val_desc = desc_vectorizer.transform(
        val_texts_desc
    )

    X_bench_desc = desc_vectorizer.transform(
        benchmark_texts_desc
    )

    desc_indices_1000, _ = sparse_topk(
        X_val_desc,
        X_items_desc,
    )

    benchmark_desc_indices, _ = sparse_topk(
        X_bench_desc,
        X_items_desc,
    )

    np.save(
        VAL_DESC_PATH,
        desc_indices_1000,
    )

    np.save(
        BENCH_DESC_PATH,
        benchmark_desc_indices,
    )

print(desc_indices_1000.shape)
print(benchmark_desc_indices.shape)

Loaded cached description TF-IDF retrieval.
(2452, 1000)
(2452, 1000)


In [43]:
desc_ids = item_ids[
    desc_indices_1000
]

desc_hits = np.array([
    relevant[i] in desc_ids[i]
    for i in range(len(validation))
])

print(
    "Description TF-IDF Recall@1000:",
    f"{desc_hits.mean():.4f}",
    desc_hits.sum(),
)

Description TF-IDF Recall@1000: 0.7276 1784


## Location prior и объединение кандидатов

Локация является сильным сигналом релевантности, но не используется как
жёсткий фильтр: некоторые поисковые location соответствуют более широким
регионам, а релевантное объявление может иметь другой `item_location_id`.

Поэтому по train оценивается:

`P(item_location_id | search_location_id)`.

Для локальной validation prior строится только по `train_fit`, чтобы избежать
утечки скрытых positive-пар. Для benchmark используется весь train.

Кандидаты из dense, word TF-IDF, char TF-IDF и description TF-IDF
объединяются в общий pool.

In [44]:
def build_location_prior(df):
    prior = (
        df
        .groupby(
            ["search_location_id", "item_location_id"]
        )
        .size()
        .reset_index(name="count")
    )

    prior["total"] = (
        prior
        .groupby("search_location_id")["count"]
        .transform("sum")
    )

    prior["prob"] = (
            prior["count"] / prior["total"]
    )

    return {
        (
            row.search_location_id,
            row.item_location_id,
        ): row.prob
        for row in prior.itertuples()
    }

In [45]:
unseen_queries = set(
    validation.loc[
        validation["split"] == "unseen",
        "search_query",
    ]
)

seen_hidden_pairs = (
    validation.loc[
        validation["split"] == "seen",
        ["search_query", "relevant_item_id"],
    ]
    .rename(columns={"relevant_item_id": "item_id"})
    .drop_duplicates()
)

train_fit = train[
    ~train["search_query"].isin(unseen_queries)
].copy()

train_fit = train_fit.merge(
    seen_hidden_pairs.assign(_hidden_pair=1),
    on=["search_query", "item_id"],
    how="left",
)

removed_seen_rows = (
    train_fit["_hidden_pair"]
    .notna()
    .sum()
)

train_fit = (
    train_fit[
        train_fit["_hidden_pair"].isna()
    ]
    .drop(columns="_hidden_pair")
    .reset_index(drop=True)
)

print("Удалено hidden seen rows:", removed_seen_rows)
print("train_fit:", train_fit.shape)

Удалено hidden seen rows: 1146
train_fit: (462092, 19)


In [46]:
location_prob = build_location_prior(
    train_fit
)

location_prob_full = build_location_prior(
    train
)

print(
    "Validation location pairs:",
    len(location_prob)
)

print(
    "Full-train location pairs:",
    len(location_prob_full)
)

Validation location pairs: 15743
Full-train location pairs: 16403


In [47]:
def build_union4(
        dense_indices,
        word_indices,
        char_indices,
        desc_indices,
        max_candidates=4000,
):
    n_queries = len(dense_indices)

    union_indices = np.full(
        (n_queries, max_candidates),
        -1,
        dtype=np.int32,
    )

    dense_rank = np.zeros(
        (n_queries, max_candidates),
        dtype=np.float32,
    )

    word_rank = np.zeros_like(dense_rank)
    char_rank = np.zeros_like(dense_rank)
    desc_rank = np.zeros_like(dense_rank)

    union_sizes = []

    rank_bonus = (
            1.0
            - np.arange(
        dense_indices.shape[1],
        dtype=np.float32,
    )
            / dense_indices.shape[1]
    )

    for i in range(n_queries):

        union = np.unique(
            np.concatenate([
                dense_indices[i],
                word_indices[i],
                char_indices[i],
                desc_indices[i],
            ])
        )

        m = len(union)

        if m > max_candidates:
            raise ValueError(
                f"Union too large: {m}"
            )

        union_sizes.append(m)
        union_indices[i, :m] = union

        dense_pos = np.searchsorted(
            union,
            dense_indices[i],
        )

        word_pos = np.searchsorted(
            union,
            word_indices[i],
        )

        char_pos = np.searchsorted(
            union,
            char_indices[i],
        )

        desc_pos = np.searchsorted(
            union,
            desc_indices[i],
        )

        dense_rank[i, dense_pos] = rank_bonus
        word_rank[i, word_pos] = rank_bonus
        char_rank[i, char_pos] = rank_bonus
        desc_rank[i, desc_pos] = rank_bonus

    return (
        union_indices,
        np.asarray(union_sizes),
        dense_rank,
        word_rank,
        char_rank,
        desc_rank,
    )

In [48]:
(
    union4_indices,
    union4_sizes,
    dense_rank4,
    word_rank4,
    char_rank4,
    desc_rank4,
) = build_union4(
    indices_ft2_1000,
    lex_indices_1000,
    char_indices_1000,
    desc_indices_1000,
)

print(
    "Validation mean union:",
    union4_sizes.mean()
)

print(
    "Validation max union:",
    union4_sizes.max()
)

Validation mean union: 2269.5330342577486
Validation max union: 3749


In [49]:
(
    union4_bench,
    union4_sizes_bench,
    dense_rank_bench,
    word_rank_bench,
    char_rank_bench,
    desc_rank_bench,
) = build_union4(
    benchmark_dense_indices,
    benchmark_lex_indices,
    benchmark_char_indices,
    benchmark_desc_indices,
)

print(
    "Benchmark mean union:",
    union4_sizes_bench.mean()
)

print(
    "Benchmark max union:",
    union4_sizes_bench.max()
)

Benchmark mean union: 2298.267128874388
Benchmark max union: 3879


In [50]:
def score_union_semantic(
        union_indices,
        union_sizes,
        query_embeddings,
        item_embeddings,
        device=DEVICE,
        batch_size=4,
):
    item_tensor = torch.from_numpy(
        item_embeddings
    ).to(device)

    query_tensor = torch.from_numpy(
        query_embeddings
    ).to(device)

    semantic = np.full(
        union_indices.shape,
        -1.0,
        dtype=np.float32,
    )

    with torch.no_grad():
        for start in range(
                0,
                len(query_embeddings),
                batch_size,
        ):
            end = min(
                start + batch_size,
                len(query_embeddings),
            )

            idx_np = union_indices[start:end]
            valid_np = idx_np >= 0

            safe_idx = idx_np.copy()
            safe_idx[~valid_np] = 0

            idx = torch.from_numpy(
                safe_idx.astype(np.int64)
            ).to(device)

            candidate_embeddings = (
                item_tensor[idx]
            )

            q = (
                query_tensor[start:end]
                .unsqueeze(1)
            )

            scores = (
                    candidate_embeddings * q
            ).sum(dim=2)

            valid = torch.from_numpy(
                valid_np
            ).to(device)

            scores[~valid] = -1.0

            semantic[start:end] = (
                scores.cpu().numpy()
            )

            del (
                idx,
                candidate_embeddings,
                q,
                scores,
                valid,
            )

    return semantic

In [51]:
semantic4 = score_union_semantic(
    union4_indices,
    union4_sizes,
    val_embeddings_ft2,
    item_embeddings_ft2,
)

print(semantic4.shape)

(2452, 4000)


In [52]:
semantic4_bench = score_union_semantic(
    union4_bench,
    union4_sizes_bench,
    benchmark_embeddings_ft2,
    item_embeddings_ft2,
)

print(semantic4_bench.shape)

(2452, 4000)


In [53]:
def score_union_location(
        union_indices,
        union_sizes,
        query_locations,
        item_locations,
        location_prior,
):
    location_score = np.zeros(
        union_indices.shape,
        dtype=np.float32,
    )

    exact_location = np.zeros(
        union_indices.shape,
        dtype=np.float32,
    )

    for i in range(len(union_indices)):
        m = union_sizes[i]

        idx = union_indices[i, :m]

        candidate_locations = (
            item_locations[idx]
        )

        search_location = query_locations[i]

        location_score[i, :m] = [
            location_prior.get(
                (
                    search_location,
                    item_location,
                ),
                0.0,
            )
            for item_location
            in candidate_locations
        ]

        exact_location[i, :m] = (
                candidate_locations
                == search_location
        ).astype(np.float32)

    return location_score, exact_location

In [54]:
item_locations = (
    items["item_location_id"]
    .to_numpy()
)

In [55]:
query_locations = (
    validation["search_location_id"]
    .to_numpy()
)

location4, exact_location4 = (
    score_union_location(
        union4_indices,
        union4_sizes,
        query_locations,
        item_locations,
        location_prob,
    )
)

In [56]:
benchmark_locations = (
    benchmark_queries[
        "search_location_id"
    ]
    .to_numpy()
)

(
    location4_bench,
    exact_location4_bench,
) = score_union_location(
    union4_bench,
    union4_sizes_bench,
    benchmark_locations,
    item_locations,
    location_prob_full,
)

In [57]:
manual4_score = (
        semantic4
        + 0.50 * location4
        + 0.005 * word_rank4
)

manual4_score_bench = (
        semantic4_bench
        + 0.50 * location4_bench
        + 0.005 * word_rank_bench
)

In [58]:
PRE_K = 300


def select_top_candidates(
        union_indices,
        scores,
        k=PRE_K,
):
    positions = np.argpartition(
        -scores,
        kth=k - 1,
        axis=1,
    )[:, :k]

    selected_scores = np.take_along_axis(
        scores,
        positions,
        axis=1,
    )

    order = np.argsort(
        -selected_scores,
        axis=1,
    )

    positions = np.take_along_axis(
        positions,
        order,
        axis=1,
    )

    indices = np.take_along_axis(
        union_indices,
        positions,
        axis=1,
    )

    return positions, indices

In [59]:
pre300_pos, pre300_indices = (
    select_top_candidates(
        union4_indices,
        manual4_score,
    )
)

print(pre300_indices.shape)

(2452, 300)


In [60]:
(
    pre300_pos_bench,
    pre300_indices_bench,
) = select_top_candidates(
    union4_bench,
    manual4_score_bench,
)

print(pre300_indices_bench.shape)

(2452, 300)


In [61]:
pre300_ids = item_ids[
    pre300_indices
]

pre300_hits = np.array([
    relevant[i] in pre300_ids[i]
    for i in range(len(validation))
])

print(
    "Preselection Recall@300:",
    f"{pre300_hits.mean():.4f}",
    f"({pre300_hits.sum()}/{len(pre300_hits)})",
)

Preselection Recall@300: 0.9139 (2241/2452)


## CatBoost pre-ranker

После recall-oriented retrieval общий candidate pool сокращается до top-300.
Финальные 50 кандидатов выбираются `CatBoostRanker`.

Ranker использует 17 признаков:

- E5 cosine similarity;
- location prior и exact location match;
- rank в dense / word / char / description retrieval;
- факт присутствия кандидата в каждом retrieval-канале;
- rating, число отзывов и price;
- availability контактных способов;
- наличие дополнительных параметров у query.

CatBoost не ищет кандидатов во всём корпусе: он только переупорядочивает
300 кандидатов, прошедших первый этап.

In [62]:
items["item_rating"] = pd.to_numeric(
    items["item_rating"],
    errors="coerce",
).fillna(0.0)

items["item_rating_reviews_count"] = pd.to_numeric(
    items["item_rating_reviews_count"],
    errors="coerce",
).fillna(0.0)

items["item_price"] = pd.to_numeric(
    items["item_price"],
    errors="coerce",
).fillna(0.0)

In [63]:
FEATURE_NAMES = [
    "semantic",
    "location_prior",
    "exact_location",

    "dense_rank",
    "word_rank",
    "char_rank",
    "desc_rank",

    "in_dense",
    "in_word",
    "in_char",
    "in_desc",

    "rating",
    "log_reviews",
    "log_price",

    "phone_hidden",
    "message_forbidden",

    "query_has_params",
]


def build_ranker_features(
        query_df,
        pre_positions,
        pre_indices,
        semantic,
        location,
        exact_location,
        dense_rank,
        word_rank,
        char_rank,
        desc_rank,
):
    def take(arr):
        return np.take_along_axis(
            arr,
            pre_positions,
            axis=1,
        )

    f_semantic = take(semantic)
    f_location = take(location)
    f_exact_location = take(exact_location)

    f_dense_rank = take(dense_rank)
    f_word_rank = take(word_rank)
    f_char_rank = take(char_rank)
    f_desc_rank = take(desc_rank)

    f_in_dense = (f_dense_rank > 0).astype(np.float32)
    f_in_word = (f_word_rank > 0).astype(np.float32)
    f_in_char = (f_char_rank > 0).astype(np.float32)
    f_in_desc = (f_desc_rank > 0).astype(np.float32)

    # Item features
    rating = (
        items["item_rating"]
        .to_numpy()[pre_indices]
        .astype(np.float32)
    )

    log_reviews = np.log1p(
        items["item_rating_reviews_count"]
        .clip(lower=0)
        .to_numpy()[pre_indices]
    ).astype(np.float32)

    log_price = np.log1p(
        items["item_price"]
        .clip(lower=0)
        .to_numpy()[pre_indices]
    ).astype(np.float32)

    phone_hidden = (
        items["item_is_phone_hidden"]
        .fillna(False)
        .astype(float)
        .to_numpy()[pre_indices]
        .astype(np.float32)
    )

    message_forbidden = (
        items["item_is_message_forbidden"]
        .fillna(False)
        .astype(float)
        .to_numpy()[pre_indices]
        .astype(np.float32)
    )

    has_params = (
        query_df["search_infm_params_text"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .astype(np.float32)
        .to_numpy()
    )

    has_params = np.repeat(
        has_params[:, None],
        PRE_K,
        axis=1,
    )

    X = np.stack(
        [
            f_semantic,
            f_location,
            f_exact_location,

            f_dense_rank,
            f_word_rank,
            f_char_rank,
            f_desc_rank,

            f_in_dense,
            f_in_word,
            f_in_char,
            f_in_desc,

            rating,
            log_reviews,
            log_price,

            phone_hidden,
            message_forbidden,

            has_params,
        ],
        axis=2,
    ).astype(np.float32)

    return X

In [64]:
X = build_ranker_features(
    query_df=validation,
    pre_positions=pre300_pos,
    pre_indices=pre300_indices,
    semantic=semantic4,
    location=location4,
    exact_location=exact_location4,
    dense_rank=dense_rank4,
    word_rank=word_rank4,
    char_rank=char_rank4,
    desc_rank=desc_rank4,
)

print(X.shape)

assert X.shape == (
    len(validation),
    PRE_K,
    17,
)

(2452, 300, 17)


In [65]:
item_ids = (
    items["item_id"]
    .astype(str)
    .to_numpy()
)

candidate_ids = item_ids[
    pre300_indices
]

relevant = (
    validation["relevant_item_id"]
    .astype(str)
    .to_numpy()
)

y = (
        candidate_ids
        == relevant[:, None]
).astype(np.int8)

print(
    "Queries with positive in top-300:",
    (y.sum(axis=1) > 0).sum(),
)

print(
    "Total positives:",
    y.sum(),
)

Queries with positive in top-300: 2241
Total positives: 2241


In [66]:
X_bench = build_ranker_features(
    query_df=benchmark_queries,
    pre_positions=pre300_pos_bench,
    pre_indices=pre300_indices_bench,
    semantic=semantic4_bench,
    location=location4_bench,
    exact_location=exact_location4_bench,
    dense_rank=dense_rank_bench,
    word_rank=word_rank_bench,
    char_rank=char_rank_bench,
    desc_rank=desc_rank_bench,
)

print("X validation:", X.shape)
print("X benchmark :", X_bench.shape)

X validation: (2452, 300, 17)
X benchmark : (2452, 300, 17)


In [72]:
CATBOOST_MODEL_PATH = Path(
    "artifacts/catboost_ranker_final.cbm"
)

if CATBOOST_MODEL_PATH.exists():

    final_model = CatBoostRanker()

    final_model.load_model(
        str(CATBOOST_MODEL_PATH)
    )

    print("Loaded cached CatBoost model.")

else:

    train_q = np.where(
        y.sum(axis=1) > 0
    )[0]

    print(
        "Training groups:",
        len(train_q),
    )

    X_train_final = X[
        train_q
    ].reshape(
        -1,
        X.shape[2],
    )

    y_train_final = y[
        train_q
    ].reshape(-1)

    group_train_final = np.repeat(
        np.arange(len(train_q)),
        PRE_K,
    )

    train_pool_final = Pool(
        X_train_final,
        label=y_train_final,
        group_id=group_train_final,
        feature_names=FEATURE_NAMES,
    )

    final_model = CatBoostRanker(
        loss_function="YetiRank",
        iterations=500,
        depth=6,
        learning_rate=0.05,
        l2_leaf_reg=5,
        random_seed=42,
        verbose=100,
        allow_writing_files=False,
    )

    final_model.fit(
        train_pool_final
    )

    final_model.save_model(
        str(CATBOOST_MODEL_PATH)
    )

Loaded cached CatBoost model.


In [68]:
bench_cat_scores = final_model.predict(
    X_bench.reshape(
        -1,
        X_bench.shape[2],
    )
).reshape(
    len(benchmark_queries),
    PRE_K,
)

print(bench_cat_scores.shape)

(2452, 300)


In [69]:
top50_pos_bench = np.argpartition(
    -bench_cat_scores,
    kth=49,
    axis=1,
)[:, :50]

top50_scores_bench = np.take_along_axis(
    bench_cat_scores,
    top50_pos_bench,
    axis=1,
)

# Для Recall порядок внутри top-50 не важен,
# но сортируем для детерминированного answer.csv.
order = np.argsort(
    -top50_scores_bench,
    axis=1,
)

top50_pos_bench = np.take_along_axis(
    top50_pos_bench,
    order,
    axis=1,
)

final_indices = np.take_along_axis(
    pre300_indices_bench,
    top50_pos_bench,
    axis=1,
)

final_ids = item_ids[
    final_indices
]

print(final_ids.shape)

(2452, 50)


In [70]:
answer = pd.DataFrame({
    "query_id": (
        benchmark_queries["query_id"]
        .astype(str)
    ),
    "answer": [
        " ".join(row)
        for row in final_ids
    ],
})

answer.to_csv(
    "answer.csv",
    index=False,
)

answer.head()

,query_id,answer
0,70DfDUpwjxB4lzFd,603623b4bd9e8f6c b92ee8f432cec2d1 255fbeaf526a...
1,JTrdTaZJvSiLPkXj,cbeccbecb1fb8d86 1c6d2079e07f4497 367af128a9ea...
2,LZCZNoVG4AFUkVRJ,d62074dd39caefee d8fce513e4f000a7 168a9207e80b...
3,660ac9QVtXkRxZC3,a846a2a5241e1180 bd2cb4500fad728e af91ec4a29b6...
4,YgHcM9MVbxKnxD1e,13f58fe9542bdb1a ba78ad593ec3412d 53f9c8caeeb8...
